In [ ]:
from dotenv import load_dotenv
import os
from pathlib import Path
workspace_root = Path.cwd().parent
env_paths = (
    Path.cwd() / ".env",
    workspace_root / ".env",
    workspace_root / "Langchain_Basics" / ".env",
)

for env_path in env_paths:
    if env_path.exists():
        load_dotenv(env_path, override=True)
        print(f"Loaded environment from: {env_path}")
        break
else:
    print("No .env file found. Create Agents/.env or workspace-root/.env.")

os.environ.setdefault("LANGSMITH_TRACING", "true")
os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
os.environ.setdefault("LANGSMITH_PROJECT", "LangChainTrainings-Agents")

if os.getenv("LANGSMITH_API_KEY"):
    print(f"LangSmith tracing enabled for project: {os.environ['LANGSMITH_PROJECT']}")
else:
    print("Add LANGSMITH_API_KEY to .env to enable LangSmith tracing.")

In [ ]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    base_url="http://localhost:11434",
    model="qwen3:8b",
    temperature=0.5,
    num_predict=2500,
)

In [ ]:
from langchain.tools import tool

@tool
def add(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b

@tool
def substarct(a: int, b: int) -> int:
    """Add two numbers."""
    return a - b

@tool
def multiply(a: int, b: int) -> int:
    """Add two numbers."""
    return a * b

print(add.invoke({"a":10, "b": 20}))  # Example usage of the custom tool

tools = [add, substarct , multiply]

llm_with_tools = llm.bind_tools(tools)

In [ ]:
response =  llm_with_tools.invoke("What is 2+3")
response

In [ ]:
# Without message state

from langchain.messages import SystemMessage, HumanMessage
from langgraph.graph import StateGraph, START, END
from typing_extensions import TypedDict
from IPython.display import display, Image

system_message = SystemMessage(
    content="You are a helpful assistant that can perform basic arithmetic operations"
)

tools_by_name = {tool.name: tool for tool in tools}


class State(TypedDict):
    question: str
    tools_name: str
    tools_args: dict
    tool_result: str
    answer: str


def assistant(state: State):
    """Ask the model. If it requests a tool, return the tool name and args."""
    response = llm_with_tools.invoke(
        [system_message, HumanMessage(content=state["question"])]
    )
    tool_calls = response.tool_calls
    if tool_calls:
        tool_call = tool_calls[0]
        return {
            "tools_name": tool_call["name"],
            "tools_args": tool_call["args"],
            "answer": "",
        }
    return {"tools_name": "", "tools_args": {}, "answer": response.content}


def tools_node(state: State):
    """Run the tool the assistant asked for and store its result."""
    tool = tools_by_name[state["tools_name"]]
    result = tool.invoke(state["tools_args"])
    return {"tool_result": str(result)}


def generate_answer(state: State):
    """Produce the final answer from the question and tool result."""
    response = llm.invoke(
        [
            system_message,
            HumanMessage(
                content=(
                    f"Question: {state['question']}\n"
                    f"The tool returned: {state['tool_result']}\n"
                    "Answer the original question."
                )
            ),
        ]
    )
    return {"answer": response.content}


def route_after_assistant(state: State):
    """Send the flow to the tools node if a tool was requested, else finish."""
    if state.get("tools_name"):
        return "tools_node"
    return END


graph = StateGraph(State)
graph.add_node("assistant", assistant)
graph.add_node("tools_node", tools_node)
graph.add_node("generate_answer", generate_answer)

graph.add_edge(START, "assistant")
graph.add_conditional_edges("assistant", route_after_assistant)
graph.add_edge("tools_node", "generate_answer")
graph.add_edge("generate_answer", END)

graph = graph.compile()


result = graph.invoke({"question": "What is 2 + 3?"})
print(result["answer"])

display(Image(graph.get_graph().draw_mermaid_png()))


In [ ]:
# Writing with messages state

from langchain.messages import SystemMessage, HumanMessage
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.prebuilt import ToolNode
from IPython.display import display, Image

system_message = SystemMessage(
    content="You are a helpful assistant that can perform basic arithmetic operations"
)

tools_by_name = {tool.name: tool for tool in tools}


def assistant(state: MessagesState):
    """Run the model, which may request a tool."""
    return {"messages": [llm_with_tools.invoke([system_message] + state["messages"])]}


def route_tools(state: MessagesState):
    """Route to the tools node if the model asked for a tool, else finish."""
    last_message = state["messages"][-1]
    if last_message.tool_calls:
        return "tools"
    return END


graph = StateGraph(MessagesState)
graph.add_node("assistant", assistant)
graph.add_node("tools", ToolNode(tools))

graph.add_edge(START, "assistant")
graph.add_conditional_edges("assistant", route_tools)
graph.add_edge("tools", "assistant")

graph = graph.compile()
display(Image(graph.get_graph().draw_mermaid_png()))


In [ ]:
from langchain.messages import HumanMessage

graph.invoke({"messages": [HumanMessage(content="What is 2 + 3?")]})
